In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as mt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler 
from sklearn.metrics import confusion_matrix,r2_score,accuracy_score,recall_score,classification_report
from scipy.stats import zscore

In [ ]:
df=pd.read_csv(r"C:\Users\NABAM AMPI\Downloads\dataset_phishing.csv")
df

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(include=['int64', 'float64']).skew()

In [ ]:
df.isnull().sum()[df.isnull().sum()>0]

### There is not any need for imputation as there are no missing values within the features in the dataset

In [ ]:
df.duplicated().sum()

### No duplicates within the dataset


In [ ]:
lr=LabelEncoder()
df['status']=lr.fit_transform(df['status'])

In [ ]:
mt.figure(figsize=(15, 10))
df.boxplot(rot=90)
mt.title("Boxplot of Features to Detect Outliers")
mt.show()


In [ ]:
df_numeric = df.select_dtypes(include=['int64', 'float64'])

Q1 = df_numeric.quantile(0.25)
Q3 = df_numeric.quantile(0.75)
IQR = Q3 - Q1
outliers = ((df_numeric < (Q1 - 1.5 * IQR)) | (df_numeric > (Q3 + 1.5 * IQR)))
outliers.sum()


In [ ]:
X=df.select_dtypes(include='number')
Y=df.iloc[:,-1]

In [ ]:
Y=df.iloc[:,-1]

In [ ]:
sl=StandardScaler()
numerical_cols = df.select_dtypes(include='number')  

In [ ]:
X.std()
X.mean()

In [ ]:
X=sl.fit_transform(X)

In [ ]:
print(X.mean())

In [ ]:
print(X.std())

In [ ]:
rs=RobustScaler()
X=rs.fit_transform(X)

In [ ]:
print(X.mean())

In [ ]:
print(X.std())

## X has been scaled to mean at 0.24 and std at 1.38 using RobustScaler as outliers exist

In [ ]:
xtrain,xtest,ytrain,ytest=train_test_split(X,Y,test_size=0.2,random_state=42)

In [ ]:
print((len(xtrain)/len(X))*100)

In [ ]:
print((len(xtest)/len(X))*100)

### Final Preprocessed Dataset

a. No missing values

b. No duplicates

c. Categorical Feature df['status'] : Encoded using LabelEncoder

d. Numerical Features : Scaled using RobustScaler

e.train_test_split :(80/20) splitting done on the datatset

### Encoding Object columns for feature selection


In [ ]:
df['url']=lr.fit_transform(df['url'])

In [ ]:
df

In [ ]:
mean=df.mean()
median=df.median()
std=df.std()


### Dropping highly correlated features

In [ ]:
corr_matrix= df.corr().abs()
to_drop=[]
for i in corr_matrix.columns:
    for j in corr_matrix.index:
        if i!=j and corr_matrix.loc[i,j]>0.9:
            if i not in to_drop and j not in to_drop:
                to_drop.append(i)


df_reduced=df.drop(columns=list(to_drop))
print(f'dropped columns : {to_drop}')

### Feature Selection

In [ ]:
X=df.iloc[:,:-1]
Y=df.iloc[:,-1]

### Train_test_split

xtrain,xtest,ytrain,ytest=train_test_split(X,Y,test_size=0.3,random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rfc=RandomForestClassifier(n_estimators=100,random_state=42)
rfc.fit(X,Y)

In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rfc.feature_importances_
}).sort_values(by='importance', ascending=False)

In [ ]:
importances.head(30)

In [ ]:
top_features = importances.head(30)['feature']
# Create a reduced dataset
X_selected = X[top_features]
X_selected

In [ ]:
rfc.fit(X_selected,Y)

In [ ]:
rfc.fit(xtest,ytest)
ypred=rfc.predict(xtest)
from sklearn.metrics import classification_report
print(classification_report(ytest,ypred))

## Feature Engineering

### Renaming columns with the word 'nb_'  and adding the prefix : 'Count of'
### Example: nb_at --> Count of at

In [ ]:
df.columns = [col.replace('nb','Count_of') for col in df.columns]
df.columns=[col.replace('length','length_of') for col in df.columns]
df.columns

# Renamed columns with the words like ratio_extHyperlinks:
# Example : ratio_intHyperlinks --> ratio_of_internal_hyperlinks

In [ ]:
df.columns=[col.replace('ratio_','ratio_of_') for col in df.columns]
df.columns=[col.replace('_ext','_external_') for col in df.columns]
df.columns=[col.replace('_int','_internal_') for col in df.columns]
df.columns=[col.replace('ratio__','ratio_of') for col in df.columns]

df.columns

In [ ]:
rfc.score(xtrain,ytrain)

In [ ]:
rfc.score(xtest,ytest)

In [ ]:
X_train_scaled = sl.fit_transform(xtrain)
X_test_scaled = sl.transform(xtest)


In [ ]:
from sklearn.decomposition import PCA
pca=PCA()
pca.fit(X_train_scaled)

In [ ]:
mt.figure(figsize=(10, 5))
mt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
mt.xlabel('Number of Components')
mt.ylabel('Cumulative Explained Variance')
mt.grid(True)
mt.title('Explained Variance vs Components')
mt.show()

## Classification Models Implementation

In [ ]:
def create_model(model):
    model.fit(X_train_scaled,ytrain)
    ypred=model.predict(X_test_scaled)
    print(confusion_matrix(ytest,ypred))
    print(classification_report(ytest,ypred))   
    return model


In [ ]:
from sklearn.linear_model import LogisticRegression
lr=create_model(LogisticRegression())


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gbc=create_model(GradientBoostingClassifier(learning_rate=0.5,n_estimators=500))

In [ ]:
from xgboost import XGBClassifier
xgb=create_model(XGBClassifier(n_estimators=500,max_depth=4,reg_alpha=0.5))

In [ ]:
from sklearn.svm import SVC
svc=create_model(SVC(C=1.0,  random_state=42))

In [ ]:

print("Train Accuracy:", accuracy_score(ytrain, xgb.predict(xtrain)))
print("Test Accuracy:", accuracy_score(ytest, xgb.predict(xtest)))


In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(xgb, X, Y, cv=5, scoring='accuracy')
print("CV Accuracy:", scores.mean())

# Analysis

## Overview 
#### 
Type: Supervised classification dataset

Rows: 11,000+

Features: 30+ (mostly numerical)

Target Column: status (1 = Phishing, 0 = Legitimate)

Data Quality: No missing values or nulls

Preprocessing Performed:

a. Label encoding of target

b. Scaling using RobustScaler

c. Checked and counted outliers using IQR method

## EDA
### 
1. Balanced Dataset for existing outliers
2. Scaled features to mean at 0.24 and std at 1.38
3. Encoded target column to numeric
4. Dropped Columns: 'nb_and', 'longest_words_raw'
5. Important features acquired with RandomForestClassifer
6. Feature Engineering done as per necessity
7. Reduced overfitting with PCA

## Conclusion
#### The trained XGBoost model delivers 97% accuracy in phishing detection. It's effective, lightweight, and suitable for integration into real-world security pipelines.


## Real-world Application

1. Email filtering
2. Cybersecurity Software
3. Browser extensions